# L1a: Values and Primitive Data Types

Every value in a computer program has a type. This lecture focuses on Julia's primitive values: how their types determine storage, interpretation, and valid operations.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Inspect a value's representation:__ Report the type, the storage width, and the bit pattern of any primitive value, and explain what each of those three answers tells you that the other two do not.
> * __Compare primitive types:__ Explain why the integer, Boolean, and floating-point families reserve different numbers of bytes, and why one stored bit pattern denotes different numbers depending on which type claims it.
> * __Connect characters to representation:__ Distinguish a character from the code point that names it and from the bytes that encode it, and explain why moving between those three is a decoding step rather than a reinterpretation of the same bits.

Let's get started!
___


## Setup, Data, and Prerequisites

Run the local setup cell first. It activates the single pinned course environment, loads every package used by this meeting, and includes any meeting source code.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This notebook does not need it; everything below uses only Julia's `Base` library. We start using the package later in the course.

___

## Primitive Data Types
Primitive data types are the basic building blocks a language provides. They are _atomic_: they are not composed of other types, and they hold simple values such as numbers, characters, and truth values.

> __Why does the type matter?__
>
> For the primitive types in this lecture, the language fixes how many bytes a value occupies, how the bit pattern in those bytes is interpreted, and which operations the compiler or interpreter will permit. Two values with identical bits can denote entirely different numbers under two different types.

We start with [Integers](https://docs.julialang.org/en/v1/base/numbers/#Core.Int) and [the `Bool` type](https://docs.julialang.org/en/v1/base/numbers/#Core.Bool), then turn to floating-point values and characters.

### Integer and Boolean Types
An __integer__ represents a whole number $x\in\mathbb{Z}$: positive, negative, or zero. Julia stores integers in a _fixed-width_ binary form, typically 32 or 64 bits. A __boolean__, which is a type that can only have the values `true` or `false`, represents a truth value and is stored in a single bit.

We will ask three questions of each: what is its type, what bits are stored, and how much space does it take. Let's bind a whole number to `x::Int64` and start there.

In [ ]:
x = 2 |> Int64; # select a whole number ... -2, -1, 0, 1, 2, ...

Every Julia value carries its own type, and [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) reports it:

In [ ]:
typeof(x) # this returns the type of the argument

The type tells us how the bits are _interpreted_. The [`bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) shows us the bits themselves.

> __Reading a bitstring:__
>
> The result has one character per bit, most significant bit first, so an `Int64` produces a 64-character string. That ordering is a fixed display convention: [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) always writes the most significant bit on the left, whatever the machine does internally. 
> 
> What you are reading is the stored bit pattern itself, not a decimal rendering of the value. [The `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) works on primitive values like `Int64` and `Float64`; hand it a `Tuple` or a `String` and it throws [an `ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError).

So what is actually stored for `x`?

In [ ]:
bitstring(x) # shows the bit pattern stored in memory

The bit pattern has to occupy space. The [`sizeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D) reports how much, measured in __bytes__:

In [ ]:
sizeof(x) # number of bytes used to store x::Int64

Now the same three questions for a boolean. A variable of type `Bool` ranges over $\mathbb{B} = \left\{\text{true},\text{false}\right\}$, so it carries exactly one bit of information. Let's bind `false` to `flag::Bool`:

In [ ]:
flag = false; # the flag variable can take on values of {true | false}

The pattern is the same as before. First the type:

In [ ]:
typeof(flag)

Then the stored bits:

In [ ]:
bitstring(flag) # this should be 8 bits wide

A `Bool` carries a single bit of information, but memory is addressed in __bytes__, so the smallest unit the machine hands out is a whole byte. Let's ask [the `sizeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.sizeof-Tuple%7BType%7D) how much space `flag` takes up, and compare it against the eight bytes we just measured for `x`:

In [ ]:
sizeof(flag) # number of bytes used to store the Bool

___

### Floating-Point Types
Floating-point types model real numbers using three components according to [the IEEE 754 standard](https://en.wikipedia.org/wiki/IEEE_754): a sign bit, an exponent (the scale), and a significand. Only the _fractional part_ of the significand is stored in memory; the leading digit is implicit, and for normalized values it is a `1`. We take a floating-point number apart bit by bit in `L1c`, where that implicit leading digit turns out to matter.

> __Julia versus Python floating point numbers__: Julia provides three standard IEEE-754 floating-point types that trade off precision for storage: `Float16` (half-precision), `Float32` (single-precision), and `Float64` (double-precision). Python's built-in `float` type is always 64-bit double-precision.

Let's look at a couple of examples. First, here's a 64-bit number (Julia's default):

In [ ]:
let
    x = 54.13; # default: in Julia, the default floating point number is 64-bit.
    bitstring(x)
end

The same decimal literal, converted to 32 bits, has a different memory layout. It is also no longer exactly the same number: `Float32` cannot pin down `54.13` as closely as `Float64` can, which is the precision story `L1c` takes up.

In [ ]:
let
    x = 54.13 |> Float32 # cast to Float32 (single precision), not Float64
    bitstring(x) # gives a string with the bit pattern
end

Fewer bits means less storage and, as we will see in `L1c`, less precision. `Float16` halves the width again:

In [ ]:
let
    x = 54.13 |> Float16 # cast to Float16 (half precision), not Float64
    sizeof(x) # returns number of bytes used to store x
end

___

### Character Types
Text on computers is composed of characters, and each character is associated with a unique integer called its __code point__. Traditional systems used [ASCII](https://en.wikipedia.org/wiki/ASCII) with one byte per character, while modern systems use [Unicode encodings like UTF-8 or UTF-16](https://en.wikipedia.org/wiki/Unicode) to represent a much wider range of characters.

> __What a `Char` actually is:__
>
> It is tempting to say characters "are" integers, but in Julia `Char <: Integer` is `false`. A [Char](https://docs.julialang.org/en/v1/base/strings/#Core.Char) is its own primitive type that _converts to and from_ integers.
>
> Character encodings define the mapping between textual symbols and numeric code points, enabling text to be stored and transmitted as bytes. Julia's `Char` is a 4-byte (32-bit) primitive, but the bits it stores are the character's __UTF-8 bytes__, left-aligned in the word, _not_ the code point. `UInt32(c)` converts to the code point; it does not simply reinterpret the bits. We will see the difference below.

Let's explore [the `Char` type in Julia](https://docs.julialang.org/en/v1/manual/unicode-input/) (notice the single quotes):

In [ ]:
c = '🍣' # example Unicode character in Julia. See: https://docs.julialang.org/en/v1/manual/unicode-input/

What is the code point (the unique integer) for the character `c`? We convert it with [the `UInt32(...)` constructor](https://docs.julialang.org/en/v1/base/numbers/#Core.UInt32):

In [ ]:
code = UInt32(c) # extract code point as UInt32 (4 bytes)

_Hmmm, what?_ That is a strange-looking result. `code` is an ordinary 32-bit unsigned integer; Julia simply __displays__ unsigned integers in [hexadecimal](https://en.wikipedia.org/wiki/Hexadecimal), i.e., base 16, and the `0x` prefix is the convention that tells you so. The same value written in base 10 is `127843`. We dig into representations in other bases in `L1c`.

__Stored bits versus code point.__ The callout at the top of this section claimed a `Char` holds UTF-8 bytes rather than the code point. Let's check that claim directly:

In [ ]:
(codepoint = string(UInt32(c), base = 16, pad = 8), # what UInt32(c) converts to
 stored     = string(reinterpret(UInt32, c), base = 16, pad = 8)) # what is actually in the 4 bytes

The two fields disagree, and that is the whole point. The code point is `0001f363`, the number Unicode assigns to this character. The 32 bits actually stored in `c` are `f09f8da3`, which is the character's UTF-8 encoding, the byte sequence `f0 9f 8d a3`, packed into the word and left-aligned. So `UInt32(c)` __decoded__ the character; it did not hand back a copy of the stored bits.

Can we see the individual bytes of a 4-byte word? Yes! Let's use [the `reinterpret(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.reinterpret) to break one into four 1-byte blocks. We'll do it on the code point:

In [ ]:
reinterpret(Tuple{UInt8, UInt8, UInt8, UInt8}, code) |> collect # split the code point (not the stored Char)

This reinterprets the 32-bit value as four 8-bit (1-byte) values. On a little-endian host the tuple comes back least significant byte first, because the low byte sits at the lowest address and tuple fields are laid out in address order.

> __Endianness:__ This ordering is [endianness](https://en.wikipedia.org/wiki/Endianness), which describes how a machine arranges the bytes of a multi-byte value in memory. Little-endian systems (most x86-64 and ARM machines) put the least significant byte first; big-endian systems put the most significant byte first. So reinterpreting `0x0001f363` as four `UInt8` values on a little-endian machine gives `[0x63, 0xf3, 0x01, 0x00]`.

__Careful:__ This is the _opposite_ of the order we saw earlier. [The `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring) always prints most significant bit first, no matter how the host arranges bytes in memory, while [the `reinterpret(...)` function](https://docs.julialang.org/en/v1/base/arrays/#Base.reinterpret) exposes that arrangement. Run the same split on `c` instead of on `code` and you get the UTF-8 bytes in that same reversed order.

Any `isbits` value can be split into bytes this way, but that does __not__ make a `Char` a collection. [`isprimitivetype(Char)`](https://docs.julialang.org/en/v1/base/base/#Base.isprimitivetype) is `true`: it has a fixed width and no independently addressable elements. Collection types are genuinely different: they group several elements under one value, and which operations they support, indexing, insertion, removal, depends on the type you pick.

___

## Looking ahead

Primitive values become useful when we organize them. In Lab `L1b`, we will choose among tuples, arrays, sets, and dictionaries, then build a custom composite type. Lecture `L1c` returns to the floating-point bit pattern and takes it apart field by field.

___


## Summary

Every Julia value carries a type that fixes how it is stored in memory and which operations are valid on it.

> __Key Takeaways:__
>
> * **Bits alone carry no meaning:** A stored pattern becomes a number only when a type declares how wide it is and how to read it, which is why one bit pattern can denote an integer under one type and a floating-point value under another.
> * **Width is a decision, not a detail:** Each primitive family buys range and precision with bytes, so choosing `Float32` over `Float64`, or `Int32` over `Int64`, is a deliberate trade whose cost is visible in the stored pattern itself.
> * **A character is not its code point:** Unicode assigns a character a number, but the machine stores that character's UTF-8 encoding, so converting a `Char` to an integer decodes it rather than handing back the bits that were sitting in memory.

Every representation question later in the course is this same question at a larger scale: what is stored, how wide is it, and under what rule is it read. The answers grow more elaborate; the three questions do not change.
___